# Notebook 5: Building a Single-Cell Aging-Clock Baseline

This notebook tests whether transcriptional information from Tabula Muris Senis cells can predict mouse age. I first create a simple mean-age baseline, then compare a Random Forest using existing PCA coordinates with a cleaner model using dimensionality reduction fitted only on training mice.

In [116]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.ensemble import RandomForestRegressor
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_absolute_error, r2_score

In [117]:
aging_data_path = Path("../data/processed/tabula_muris_senis_dev_subset.h5ad")

print("File exists:", aging_data_path.exists())

aging_data = sc.read_h5ad(aging_data_path)

print(aging_data)

File exists: False


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '../data/processed/tabula_muris_senis_dev_subset.h5ad', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
columns_needed_for_model = [
    "age",
    "age_months",
    "mouse.id",
    "tissue",
    "cell_ontology_class"
]

aging_data.obs[columns_needed_for_model].head()

The important target column here is age_months, because ML regression models need numeric labels. I will also keep mouse.id because cells from the same mouse are biologically related, so I should avoid mixing the same mouse across training and testing if possible.

In [ ]:
print("Age groups:")
print(aging_data.obs["age"].value_counts())

print("\nNumeric age labels:")
print(aging_data.obs["age_months"].value_counts().sort_index())

In [ ]:
print("Number of unique mice:")
print(aging_data.obs["mouse.id"].nunique())

print("\nCells per mouse:")
print(aging_data.obs["mouse.id"].value_counts())

## Preparing the model input features

For this aging clock model, I am using the PCA representation already stored in the Tabula Muris Senis AnnData object.

I am using PCA features instead of raw gene expression because the dataset has more than 20K genes, which would be too high dimensional and noisy for a simple model. PCA compresses the main transcriptional patterns into fewer features making the model easier to train.

In [ ]:
cell_pca_features = aging_data.obsm["X_pca"]

cell_age_months = aging_data.obs["age_months"].astype(float).values

mouse_ids = aging_data.obs["mouse.id"].values

print("Cell PCA feature shape:", cell_pca_features.shape)
print("Age label shape:", cell_age_months.shape)
print("Mouse ID shape:", mouse_ids.shape)

One limitation of this baseline is that I am using the PCA coordinates already included in the processed dataset. This PCA was likely computed before my train/test split, so this is not a fully strict machine learning pipeline. However, I am treating this as an exploratory baseline model. 
A stricter version is created below to fit PCA only on the training cells and then apply the same PCA transformation to the test cells.

In [ ]:
mouse_age_table = aging_data.obs[
    ["mouse.id", "age", "age_months"]
].drop_duplicates()

print("Mouse-level age table:")
display(mouse_age_table)

print("\nNumber of mice per age group:")
print(mouse_age_table.groupby("age", observed=True)["mouse.id"].nunique())

I added observed=True to tell pandas to only include age groups that actually exist in the data. This silences a warning about future behaviour changes in pandas.

In [ ]:
random_generator = np.random.default_rng(42)

test_mouse_ids = []

for age_group, mice_from_this_age in mouse_age_table.groupby("age", observed=True):
    available_mice = mice_from_this_age["mouse.id"].unique()
    
    if len(available_mice) == 1:
        print(f"Only one mouse found for age {age_group}, keeping it in training.")
        continue
    
    number_of_test_mice = max(1, int(round(len(available_mice) * 0.25)))
    
# making sure at least 1 mouse remains for training
    number_of_test_mice = min(number_of_test_mice, len(available_mice) - 1)
    
    chosen_test_mice = random_generator.choice(
        available_mice,
        size=number_of_test_mice,
        replace=False
    )
    
    test_mouse_ids.extend(chosen_test_mice)

print("Test mice selected:")
print(test_mouse_ids)

I selected test mice separately within each age group so that the test set contains cells from all age groups(3m, 18m, 24m). This will ensure model is not trained on only one age group or misses out any age group. 

To reduce data leakage, I split the dataset by `mouse.id` instead of randomly splitting individual cells. This makes sure that cells from the same mouse are not present in both the training and testing sets.

In [ ]:
test_cell_mask = aging_data.obs["mouse.id"].isin(test_mouse_ids)
train_cell_mask = ~test_cell_mask

train_cell_features = cell_pca_features[train_cell_mask]
test_cell_features = cell_pca_features[test_cell_mask]

train_cell_ages = cell_age_months[train_cell_mask]
test_cell_ages = cell_age_months[test_cell_mask]

training_metadata = aging_data.obs.loc[train_cell_mask].copy()
testing_metadata = aging_data.obs.loc[test_cell_mask].copy()

print("Training cells:", train_cell_features.shape[0])
print("Testing cells:", test_cell_features.shape[0])

In [ ]:
print("Training age distribution:")
print(training_metadata["age"].value_counts())

print("\nTesting age distribution:")
print(testing_metadata["age"].value_counts())

i first tried a basic mouse based train/test split, but the test set did not contain any 24 month old cells. Since 24 months is the oldest age group, this would make the evaluation incomplete. I therefore changed the split so that test mice are selected within each age group.

## Mean Age Baseline

The simplest possible model predicts the same age for every test cell: the average age of the training cells. This gives a reference point. Any real model should beat this.

In [ ]:
average_training_age = train_cell_ages.mean()

baseline_age_predictions = np.repeat(
    average_training_age,
    len(test_cell_ages)
)

baseline_mae = mean_absolute_error(
    test_cell_ages,
    baseline_age_predictions
)

baseline_r2 = r2_score(
    test_cell_ages,
    baseline_age_predictions
)

print("Baseline mean-age model")
print("Mean absolute error:", baseline_mae)
print("R2 score:", baseline_r2)

## Stricter baseline: fitting dimensionality reduction after train/test split

The previous model used PCA coordinates already stored in the AnnData object. To make the evaluation cleaner, I now fit dimensionality reduction only on the training cells and then apply it to the test cells.

In [ ]:

highly_variable_gene_mask = aging_data.var["highly_variable"].to_numpy()

train_cell_mask_array = np.asarray(train_cell_mask, dtype=bool)
test_cell_mask_array = np.asarray(test_cell_mask, dtype=bool)


train_gene_expression = aging_data[
    train_cell_mask_array,
    highly_variable_gene_mask
].X

test_gene_expression = aging_data[
    test_cell_mask_array,
    highly_variable_gene_mask
].X

print("Training gene expression shape:", train_gene_expression.shape)
print("Testing gene expression shape:", test_gene_expression.shape)

Although SVD is fitted only on training cells, the highly variable genes were identified in the original processed AnnData object using the complete dataset. Therefore, this pipeline reduces preprocessing leakage but is not yet fully leakage-free. A fully strict pipeline would identify highly variable genes using training data only.

In [ ]:

svd_model = TruncatedSVD(
    n_components=50,
    random_state=42
)

train_reduced_features = svd_model.fit_transform(
    train_gene_expression
)

test_reduced_features = svd_model.transform(
    test_gene_expression
)

print("Training reduced feature shape:", train_reduced_features.shape)
print("Testing reduced feature shape:", test_reduced_features.shape)

print(
    "Variance represented by 50 components:",
    svd_model.explained_variance_ratio_.sum()
)

The first 50 SVD components represent approximately 23.6% of the variation in the selected gene-expression matrix. This indicates that the reduced features retain part of the transcriptional structure, although a substantial amount of variation remains outside these components.

In [ ]:
stricter_aging_clock_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

stricter_aging_clock_model.fit(
    train_reduced_features,
    train_cell_ages
)

print("Stricter Random Forest model trained.")

I used Random Forest as the first model because it is a simple baseline that works well with tabular biological data. It can capture non-linear relationships and does not require the assumptions of ordinary linear regression.

In [ ]:
# This is the first (less strict) model — it uses pre-built PCA coordinates
# rather than dimensionality reduction fitted only on training cells.

aging_clock_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

aging_clock_model.fit(train_cell_features, train_cell_ages)

print("Random Forest (precomputed PCA features) trained.")

In [ ]:
predicted_cell_ages = aging_clock_model.predict(test_cell_features)

model_mae = mean_absolute_error(test_cell_ages, predicted_cell_ages)
model_r2 = r2_score(test_cell_ages, predicted_cell_ages)

print("Precomputed PCA + Random Forest")
print("Mean absolute error:", model_mae)
print("R² score:", model_r2)

i added the training step for my first Random Forest, which uses the PCA coordinates that were already built into the processed dataset. I had been predicting with this model but had forgotten to include the line that actually creates and trains it. I also compute MAE and R² here so they can be used in the comparison table later.

In [ ]:
stricter_predicted_ages = stricter_aging_clock_model.predict(
    test_reduced_features
)

stricter_model_mae = mean_absolute_error(
    test_cell_ages,
    stricter_predicted_ages
)

stricter_model_r2 = r2_score(
    test_cell_ages,
    stricter_predicted_ages
)

print("Train-fitted SVD + Random Forest")
print("Mean absolute error:", stricter_model_mae)
print("R² score:", stricter_model_r2)

In [ ]:
model_comparison = pd.DataFrame({
    "model": [
        "Mean-age baseline",
        "Precomputed PCA + Random Forest",
        "Train-fitted SVD + Random Forest"
    ],
    "cell_level_mae": [
        baseline_mae,
        model_mae,
        stricter_model_mae
    ],
    "cell_level_r2": [
        baseline_r2,
        model_r2,
        stricter_model_r2
    ]
})

model_comparison.round(3)



Both Random Forest models performed better than the mean-age baseline at the cell level. The precomputed-PCA model produced the lowest MAE and highest R², but its PCA representation was created using the complete dataset before the train/test split.

The train-fitted-SVD model produced slightly weaker results, with an MAE of approximately 6.11 months and an R² of approximately 0.20. However, its dimensionality reduction was fitted using training cells only, making it a cleaner evaluation of generalization to unseen mice.

These metrics indicate modest predictive performance. They should be interpreted cautiously because the test cells come from only four mice and are not independent biological samples.


In [ ]:
stricter_prediction_results = pd.DataFrame({
    "actual_age_months": test_cell_ages,
    "predicted_age_months": stricter_predicted_ages
})

stricter_prediction_results.groupby(
    "actual_age_months"
)["predicted_age_months"].describe()



Mean predicted age increased across the 3-month, 18-month, and 24-month groups, suggesting that the model detected a broad age-associated transcriptional pattern.

However, the prediction distributions overlap, and the youngest cells are predicted considerably older than their true age. This compression toward intermediate ages may reflect regression toward the training-set average and variation associated with cell type, sex, tissue, or individual mice.

Because these potential confounders have not yet been controlled or interpreted at the gene level, the results should be treated as a preliminary age-prediction baseline rather than evidence of a robust biological aging clock.


In [ ]:
mouse_level_results = testing_metadata[
    ["mouse.id", "age_months"]
].copy()

mouse_level_results["predicted_age_months"] = (
    stricter_predicted_ages
)

mouse_level_summary = (
    mouse_level_results
    .groupby("mouse.id", observed=True)
    .agg(
        actual_age_months=("age_months", "first"),
        predicted_age_months=("predicted_age_months", "mean"),
        number_of_cells=("predicted_age_months", "size")
    )
    .reset_index()
)

mouse_level_summary

In [ ]:
assert not mouse_level_summary[
    ["actual_age_months", "predicted_age_months"]
].isna().any().any()

print("No missing values in the mouse level summary")

In [ ]:
mouse_level_summary["absolute_error_months"] = (
    mouse_level_summary["actual_age_months"].astype(float)
    - mouse_level_summary["predicted_age_months"].astype(float)
).abs()

mouse_level_mae = mouse_level_summary[
    "absolute_error_months"
].mean()

print("Mouse-level MAE:", mouse_level_mae)

mouse_level_summary

The mouse level MAE is approximately 6.74 months. This result is based on only four test mice, so it is useful as a preliminary check but is not a stable estimate of performance. Leave one mouse out cross validation would provide a stronger evaluation across all available mice.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

ax.scatter(
    mouse_level_summary["actual_age_months"],
    mouse_level_summary["predicted_age_months"],
    color="steelblue",
    s=80,
    zorder=3
)

# Label each dot with its mouse ID
for _, row in mouse_level_summary.iterrows():
    ax.annotate(
        row["mouse.id"],
        xy=(row["actual_age_months"], row["predicted_age_months"]),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=9
    )

# Diagonal line = perfect prediction
ax.plot([0, 26], [0, 26], linestyle="--", color="gray", label="Perfect prediction")

ax.set_xlim(0, 28)
ax.set_ylim(0, 28)
ax.set_xlabel("Actual mouse age (months)")
ax.set_ylabel("Mean predicted age (months)")
ax.set_title("Mouse-level predicted vs actual age")
ax.legend()

plt.tight_layout()
plt.show()

 I added mouse ID labels to each point so readers can see which mouse corresponds to which dot. I also added a legend to explain that the dashed line represents perfect prediction — points below the line are over-predicted as younger, and points above it are over-predicted as older.